# 09p — Uji Bentuk Pertanyaan pada Set Pengembangan (Kaggle)

**Kenapa notebook ini ada.** Pilot prompt v4 (40 chip) memberi jawaban **konstan**:
Qwen menjawab `hancur` untuk 39/40 bangunan, InternVL menjawab `tidak_rusak` untuk 40/40,
dan keempat fitur bernilai sama untuk semua bangunan. Model menjawab dari teks prompt, bukan
dari citra. Uji kendali 23 Sep (Qwen P3 3/3 benar) terlalu kecil (n = 3) untuk memastikan
sebaliknya.

Notebook ini menguji **sekumpulan kecil bentuk pertanyaan yang ditetapkan di awal** pada
**set pengembangan**: 15 bangunan (5 per kelas) dari sampel besar yang **bukan** anggota
sampel kecil evaluator. Ke-15 bangunan ini dicatat dan **dikeluarkan dari perhitungan metrik
akhir**, supaya pemilihan prompt tidak menyentuh data uji.

| Kode | Masukan | Bentuk |
|---|---|---|
| `K0` | pasca | prompt kelas v4 (acuan) |
| `K1` | pasca | satu kata: UTUH / RUSAK / HANCUR |
| `K1r` | pasca | sama dengan K1, urutan pilihan dibalik (uji bias posisi) |
| `K1s` | pasca | **skor peluang** tiap pilihan K1 (hanya Qwen): apakah model *melihat* walau tidak *mengucapkan*? |
| `F-<fitur>` | pra + pasca | satu pertanyaan pendek per fitur, jawab satu pilihan |
| `D1` | pra + pasca | deskripsi bebas satu kalimat |

Hasil disimpan sebagai run berjenis `pengembangan` di `runs/`, tidak dicampur dengan run inferensi.

Prasyarat sama dengan notebook 09 (GPU T4, Internet On, dataset chip + paket).
Kedua model dijalankan berurutan dalam satu sesi, **± 15 menit**.

## Sel 1 — pustaka

In [ ]:
!pip install -q "transformers==4.51.3" accelerate timm einops sentencepiece
import sys, torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU tidak aktif"

## Sel 2 — data, set pengembangan, dan bentuk pertanyaan

In [ ]:
import json, hashlib, importlib.util, datetime, re, time, gc
from pathlib import Path
import pandas as pd
from PIL import Image

INPUT, KERJA = Path("/kaggle/input"), Path("/kaggle/working")
def cari_satu(pola):
    h = sorted(INPUT.rglob(pola))
    if not h: raise FileNotFoundError(pola)
    return h[-1]

F_KONFIG, F_PARSER = cari_satu("konfigurasi_fitur.json"), cari_satu("05_parser_keluaran.py")
KONFIG = json.loads(F_KONFIG.read_text(encoding="utf-8"))
MP = json.loads(cari_satu("manifes_v*_*.json").read_text(encoding="utf-8"))
PROMPT_V4_KELAS = cari_satu(MP["prompt"]["kelas"]["berkas"]).read_text(encoding="utf-8")
spec = importlib.util.spec_from_file_location("parser05", F_PARSER)
P = importlib.util.module_from_spec(spec); spec.loader.exec_module(P)

besar = pd.read_csv(cari_satu("sampel_besar.csv"))
kecil = pd.read_csv(cari_satu("sampel_kecil.csv"))
calon = besar[~besar["uid"].isin(kecil["uid"])]
DEV = pd.concat([g.sample(5, random_state=42) for _, g in calon.groupby("kelas_xbd")]).reset_index(drop=True)
peta = {f.name: f for f in INPUT.rglob("*.png") if f.parent.name in ("pra", "pasca")}
DEV["f_pra"], DEV["f_pasca"] = DEV["chip_pra"].map(peta), DEV["chip_pasca"].map(peta)
assert DEV[["f_pra", "f_pasca"]].notna().all().all()
DEV["label"] = DEV["kelas_xbd"].map(KONFIG["kelas"]["pemetaan_xbd"])
print(DEV["kelas_xbd"].value_counts().to_dict())

PENANDA = "bangunan yang ditunjuk bidik silang magenta di tengah citra"
KATA_KELAS = {"UTUH": "tidak_rusak", "RUSAK": "rusak", "HANCUR": "hancur"}
TANYA = {
 "K1":  ("pasca", f"Ini citra satelit sesudah bencana. Perhatikan {PENANDA}. "
                  "Bagaimana kondisi bangunan itu? Jawab hanya dengan satu kata: UTUH, RUSAK, atau HANCUR."),
 "K1r": ("pasca", f"Ini citra satelit sesudah bencana. Perhatikan {PENANDA}. "
                  "Bagaimana kondisi bangunan itu? Jawab hanya dengan satu kata: HANCUR, RUSAK, atau UTUH."),
}
BATANG = {
 "keutuhan_atap":  "Bandingkan atap {p} pada citra pertama dan citra kedua. Bagaimana atap itu pada citra kedua?",
 "puing_sekitar":  "Perhatikan sekeliling {p} pada citra kedua. Bagaimana sebaran puing di sekitarnya?",
 "wujud_bangunan": "Bandingkan bentuk dan garis batas {p} pada citra pertama dan citra kedua. Bagaimana wujudnya pada citra kedua?",
 "sedimen_lumpur": "Perhatikan {p} dan halamannya pada citra kedua. Seberapa luas lumpur atau sedimen yang menutupinya?",
}
PEMBUKA_DUA = "Citra pertama diambil sebelum bencana, citra kedua sesudah bencana, pada lokasi yang sama. "
peta_kandidat = {c["kunci"]: c for c in KONFIG["kandidat"]}
for k in KONFIG["seleksi"]["terpilih"]:
    lv = peta_kandidat[k]["level"]
    pilihan = ", ".join(x.upper() for x in lv[:-1]) + ", atau " + lv[-1].upper()
    TANYA[f"F-{k}"] = ("dua", PEMBUKA_DUA + BATANG[k].format(p=PENANDA) +
                       f" Jawab hanya dengan salah satu: {pilihan}.")
TANYA["D1"] = ("dua", PEMBUKA_DUA + f"Dalam satu kalimat bahasa Indonesia, jelaskan kondisi {PENANDA} "
                      "pada citra kedua dibandingkan citra pertama.")
for k, (m, t) in TANYA.items():
    print(f"{k:<18}{m:<6}{t[:110]}...")

## Sel 3 — fungsi model (muat, tanya, skor peluang, lepas)

In [ ]:
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import (Qwen2_5_VLForConditionalGeneration, AutoProcessor,
                          AutoModel, AutoTokenizer)
REPO = {"qwen2.5-vl-3b": "Qwen/Qwen2.5-VL-3B-Instruct", "internvl3-2b": "OpenGVLab/InternVL3-2B"}
buka = lambda f: Image.open(f).convert("RGB")

def muat(nama):
    if nama.startswith("qwen"):
        m = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            REPO[nama], torch_dtype=torch.float16, device_map={"": 0}).eval()
        return {"nama": nama, "m": m, "proc": AutoProcessor.from_pretrained(REPO[nama])}
    try:
        m = AutoModel.from_pretrained(REPO[nama], torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
                                      use_flash_attn=False, trust_remote_code=True).eval().cuda()
    except AttributeError:
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = property(
            lambda self: list(getattr(self, "_tied_weights_keys", None) or []))
        m = AutoModel.from_pretrained(REPO[nama], torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
                                      use_flash_attn=False, trust_remote_code=True).eval().cuda()
    return {"nama": nama, "m": m,
            "tok": AutoTokenizer.from_pretrained(REPO[nama], trust_remote_code=True, use_fast=False)}

TF = T.Compose([T.Resize((448, 448), interpolation=InterpolationMode.BICUBIC), T.ToTensor(),
                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

def tanya(M, gambar, teks, maks=96):
    if M["nama"].startswith("qwen"):
        proc, m = M["proc"], M["m"]
        isi = [{"type": "image", "image": g} for g in gambar] + [{"type": "text", "text": teks}]
        chat = proc.apply_chat_template([{"role": "user", "content": isi}], tokenize=False,
                                        add_generation_prompt=True)
        x = proc(text=[chat], images=gambar, return_tensors="pt", padding=True).to(m.device)
        with torch.inference_mode():
            y = m.generate(**x, max_new_tokens=maks, do_sample=False)
        return proc.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True)
    px = [TF(g).unsqueeze(0).to(torch.bfloat16).cuda() for g in gambar]
    awal = "<image>\n" if len(px) == 1 else "".join(f"Citra-{i}: <image>\n" for i in range(1, len(px) + 1))
    return M["m"].chat(M["tok"], torch.cat(px, 0), awal + teks,
                       dict(max_new_tokens=maks, do_sample=False), num_patches_list=[1] * len(px))

def skor_opsi_qwen(M, gambar, teks, opsi):
    # log-peluang total token jawaban untuk setiap pilihan, diberi prompt dan citra yang sama
    proc, m = M["proc"], M["m"]
    isi = [{"type": "image", "image": g} for g in gambar] + [{"type": "text", "text": teks}]
    awal = proc.apply_chat_template([{"role": "user", "content": isi}], tokenize=False,
                                    add_generation_prompt=True)
    id_awal = proc(text=[awal], images=gambar, return_tensors="pt")["input_ids"][0]
    n_awal = len(id_awal)
    hasil = {}
    for o in opsi:
        x = proc(text=[awal + o], images=gambar, return_tensors="pt").to(m.device)
        ids = x["input_ids"][0]
        # awalan harus tertokenisasi identik; bila tidak, skor pilihan ini tidak sah
        if not torch.equal(ids[:n_awal].cpu(), id_awal):
            hasil[o] = float("nan"); continue
        with torch.inference_mode():
            lp = torch.log_softmax(m(**x).logits[0].float(), -1)
        sasaran = ids[n_awal:]
        hasil[o] = lp[n_awal - 1:-1].gather(1, sasaran.unsqueeze(1)).sum().item()
    return hasil

def lepas(M):
    del M["m"]; gc.collect(); torch.cuda.empty_cache()

## Sel 4 — jalankan kedua model

In [ ]:
MULAI = datetime.datetime.now(datetime.timezone.utc)
RUN_ID = f"{MULAI:%Y%m%d-%H%M%S}_dev_bentuk-pertanyaan_v{MP['versi_prompt']}_{MP['sidik_fitur']}"
DIR = KERJA / "runs" / RUN_ID; DIR.mkdir(parents=True)
F = DIR / "mentah.jsonl"
BARIS = []
for nama in ("qwen2.5-vl-3b", "internvl3-2b"):
    M = muat(nama); print(nama, "dimuat")
    for r in DEV.itertuples(index=False):
        g_pasca, g_dua = [buka(r.f_pasca)], [buka(r.f_pra), buka(r.f_pasca)]
        tugas = [("K0", g_pasca, PROMPT_V4_KELAS, 64)] + \
                [(k, g_pasca if m_ == "pasca" else g_dua, t, 96 if k == "D1" else 24)
                 for k, (m_, t) in TANYA.items()]
        for kode, gbr, teks, maks in tugas:
            t0 = time.time()
            b = {"model": nama, "uid": r.uid, "kelas_xbd": r.kelas_xbd, "label": r.label,
                 "varian": kode, "mentah": tanya(M, gbr, teks, maks), "detik": round(time.time() - t0, 2)}
            BARIS.append(b)
        if nama.startswith("qwen"):
            s = skor_opsi_qwen(M, g_pasca, TANYA["K1"][1], list(KATA_KELAS))
            BARIS.append({"model": nama, "uid": r.uid, "kelas_xbd": r.kelas_xbd, "label": r.label,
                          "varian": "K1s", "mentah": json.dumps(s), "detik": None})
        print(f"  {r.kelas_xbd:<13} selesai")
    lepas(M)
with F.open("w", encoding="utf-8") as fh:
    for b in BARIS: fh.write(json.dumps(b, ensure_ascii=False) + "\n")
(DIR / "manifest.json").write_text(json.dumps({
    "run_id": RUN_ID, "jenis": "pengembangan", "status": "selesai",
    "waktu_mulai_utc": MULAI.isoformat(timespec="seconds"),
    "tujuan": "memilih bentuk pertanyaan setelah pilot v4 memberi jawaban konstan",
    "uid_set_pengembangan": DEV["uid"].tolist(),
    "aturan": "uid set pengembangan DIKELUARKAN dari metrik akhir",
    "varian": {k: v[1] for k, v in TANYA.items()},
    "lingkungan": {"torch": torch.__version__, "transformers": transformers.__version__,
                   "gpu": torch.cuda.get_device_name(0)},
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("tersimpan:", DIR)

## Sel 5 — baca hasil

Yang dicari **bukan akurasi tinggi**, melainkan **apakah jawaban berubah mengikuti kelas**.
Kolom `unik` = banyaknya jawaban berbeda. Nilai 1 berarti jawaban konstan (tidak melihat).

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
df = pd.DataFrame(BARIS)

def ke_kelas(teks, varian):
    if varian == "K0":
        return P.urai_kelas(teks, KONFIG)["kelas_kerusakan"]
    kata = set(re.findall(r"[A-Z_]+", (teks or "").upper()))
    hit = [KATA_KELAS[k] for k in KATA_KELAS if k in kata]
    return hit[0] if len(hit) == 1 else None

def ke_fitur(teks, kunci):
    return P.selaraskan(re.sub(r"[^\w ]", "", (teks or "")).strip(), peta_kandidat[kunci]["enum"])[0]

L = []
for (m, v), g in df[df["varian"].isin(["K0", "K1", "K1r"])].groupby(["model", "varian"]):
    pred = g.apply(lambda r: ke_kelas(r["mentah"], v), axis=1)
    L.append(f"\n{m} {v}  | unik={pred.nunique(dropna=False)}  | akurasi seimbang="
             f"{balanced_accuracy_score(g['label'], pred.fillna('GAGAL')):.2f}")
    L.append(pd.crosstab(g["label"], pred.fillna("GAGAL")).to_string())
for (m, v), g in df[df["varian"].str.startswith("F-")].groupby(["model", "varian"]):
    kunci = v[2:]
    pred = g["mentah"].apply(lambda t: ke_fitur(t, kunci))
    L.append(f"\n{m} {v}  | unik={pred.nunique(dropna=False)}")
    L.append(pd.crosstab(g["label"], pred.fillna("GAGAL")).to_string())
s = df[df["varian"] == "K1s"].copy()
if len(s):
    sk = pd.DataFrame([json.loads(t) for t in s["mentah"]], index=s.index)
    pr = np.exp(sk.sub(sk.max(axis=1), axis=0)); pr = pr.div(pr.sum(axis=1), axis=0)
    s = pd.concat([s, pr], axis=1)
    L.append("\nQwen K1s: rerata peluang tiap pilihan menurut label xBD")
    L.append(s.groupby("label")[list(KATA_KELAS)].mean().round(3).to_string())
    biner = s[s["label"].isin(["tidak_rusak", "hancur"])]
    L.append(f"AUC P(HANCUR), hancur vs tidak_rusak: "
             f"{roc_auc_score(biner['label'] == 'hancur', biner['HANCUR']):.2f}  (0.5 = tidak membedakan)")
    L.append(f"AUC P(UTUH),  tidak_rusak vs rusak+hancur: "
             f"{roc_auc_score(s['label'] == 'tidak_rusak', s['UTUH']):.2f}")
    L.append(f"akurasi seimbang argmax skor: "
             f"{balanced_accuracy_score(s['label'], pr.idxmax(axis=1).map(KATA_KELAS)):.2f}")
L.append("\nContoh deskripsi D1:")
for m, g in df[df["varian"] == "D1"].groupby("model"):
    for r in g.groupby("label").head(1).itertuples():
        L.append(f"  {m[:6]} {r.label:<12} {r.mentah.strip()[:160]}")
teks = "\n".join(L); print(teks)
(DIR / "ringkasan.txt").write_text(teks, encoding="utf-8")
import shutil
print("\narsip:", shutil.make_archive(str(KERJA / f"run_{RUN_ID}"), "zip", root_dir=KERJA / "runs", base_dir=RUN_ID))